In [ ]:
!pip install /kaggle/input/litmodels-bolts/lightning_bolts-0.3.3-py3-none-any.whl

!pip install /kaggle/input/kerasapplications -q
!pip install /kaggle/input/efficientnet-keras-source-code/ -q --no-deps

In [ ]:
!conda install '/kaggle/input/pydicom-conda-helper/libjpeg-turbo-2.1.0-h7f98852_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/libgcc-ng-9.3.0-h2828fa1_19.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/gdcm-2.8.9-py37h500ead1_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/conda-4.10.1-py37h89c1867_0.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/certifi-2020.12.5-py37h89c1867_1.tar.bz2' -c conda-forge -y
!conda install '/kaggle/input/pydicom-conda-helper/openssl-1.1.1k-h7f98852_0.tar.bz2' -c conda-forge -y

# References

- https://www.kaggle.com/illidan7/siim-covid-19-classification-train
- https://www.kaggle.com/illidan7/litmodelscolab-evaluation/data
- https://www.kaggle.com/shivanandmn/efficientnet-pytorch-lightning-train-inference
- https://www.kaggle.com/illidan7/siim-covid-19-frankenstein/notebook
- https://www.kaggle.com/whitegg/inference-studyclass-baseline

# Load libraries

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.metrics.functional import accuracy
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
import platform
import numpy as np 
import pandas as pd 
import os
from tqdm.notebook import tqdm
import cv2
import pydicom
import gdcm
import glob
import gc
from math import ceil
import matplotlib.pyplot as plt
from pydicom.pixel_data_handlers.util import apply_voi_lut
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

import albumentations

# from sklearn.model_selection import StratifiedKFold, GroupKFold

import math
import psutil

DATA_PATH = '/kaggle/input/siim-covid19-detection/'

import sys
sys.path.append('../input/timm-pytorch-image-models/pytorch-image-models-master')
sys.path.append('../usr/lib/siim_covid19_littrainers_py/')

import timm
import siim_covid19_littrainers_py as lit

In [ ]:
from PIL import Image

import torch
torch.manual_seed(0)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# import timm

import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
# from pytorch_metric_learning import losses
from torch.optim import lr_scheduler

import warnings
warnings.simplefilter('ignore')

# Config

In [ ]:
class Config:
    train_pcent = 0.85
    model_name = 'resnet50'
    image_size = (400, 400)
    num_classes = 4
    batch_size = 32
    num_workers = 8
    NB_EPOCHS = 30
    scaler = GradScaler()
    scheduler = 'CosineAnnealingLR'
    weight_decay = 1e-6
    T_max = 10
    min_lr = 1e-6
    lr = 1e-5

# Dataset - Colab

In [ ]:
def dicom2array(path, voi_lut=True, fix_monochrome=True):
    dicom = pydicom.read_file(path)
    # VOI LUT (if available by DICOM device) is used to
    # transform raw DICOM data to "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
    return data

In [ ]:
class SIIMData(Dataset):
    def __init__(self, df, is_train=True, augments=None, img_size=Config.image_size):
        super().__init__()
        self.df = df.sample(frac=1).reset_index(drop=True)
        self.is_train = is_train
        self.augments = augments
        self.img_size = img_size
        
    def __getitem__(self, idx):
        image_id = self.df['StudyInstanceUID'].values[idx]
        
        image_path = self.df['path'].values[idx]
        image = dicom2array(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = cv2.resize(image, Config.image_size)
        
        # Augments must be albumentations
        if self.augments:
            image = self.augments(image=image)['image']
        else:
            image = torch.tensor(image)
        
        if self.is_train:
            label = self.df[self.df['StudyInstanceUID'] == image_id].values.tolist()[0][3:7].index(1)
            return image, torch.tensor(label)
        
        return image
    
    def __len__(self):
        return len(self.df)

# Dataset - Efnb7

In [ ]:
import numpy as np
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def read_xray(path, voi_lut = True, fix_monochrome = True):
    # Original from: https://www.kaggle.com/raddar/convert-dicom-to-np-array-the-correct-way
    dicom = pydicom.read_file(path)
    
    # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to 
    # "human-friendly" view
    if voi_lut:
        data = apply_voi_lut(dicom.pixel_array, dicom)
    else:
        data = dicom.pixel_array
               
    # depending on this value, X-ray may look inverted - fix that:
    if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
        
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)
        
    return data

In [ ]:
def resize(array, size, keep_ratio=False, resample=Image.LANCZOS):
    # Original from: https://www.kaggle.com/xhlulu/vinbigdata-process-and-resize-to-image
    im = Image.fromarray(array)
    
    if keep_ratio:
        im.thumbnail((size, size), resample)
    else:
        im = im.resize((size, size), resample)
    
    return im

In [ ]:
split = 'train'
dirs = []
files = []


for dirname, _, filenames in tqdm(os.walk(f'../input/siim-covid19-detection/{split}')):
    dirs.append(dirname)
    for file in filenames:
        files.append(file)

print(f"Number of dirs: {len(dirs)}")
print(f"Number of files: {len(files)}")

In [ ]:
split = 'train'
save_dir = f'/kaggle/tmp/{split}/'

os.makedirs(save_dir, exist_ok=True)

save_dir = f'/kaggle/tmp/{split}/study/'
os.makedirs(save_dir, exist_ok=True)

for dirname, _, filenames in tqdm(os.walk(f'../input/siim-covid19-detection/{split}'), total=len(dirs)):
    for file in filenames:
        # set keep_ratio=True to have original aspect ratio
        xray = read_xray(os.path.join(dirname, file))
        im = resize(xray, size=600)  
        study = dirname.split('/')[-2] + '_study.png'
        im.save(os.path.join(save_dir, study))

# Inference data loader

In [ ]:
!ls /kaggle/tmp/train/study/ | wc -l

In [ ]:
import os

files = []

for file in os.listdir('/kaggle/tmp/train/study/'):
    files.append(file.split('_')[0])

len(files)

In [ ]:
files[0:5]

In [ ]:
train[train['StudyInstanceUID'].isin(files)].shape

In [ ]:
# temp = pd.read_csv("../input/siim-covid19-detection/train_study_level.csv")

# temp['StudyInstanceUID'] = temp['id'].apply(lambda x: x.replace('_study', ''))
# temp = temp[~temp['StudyInstanceUID'].str.endswith('_image')]

# temp['path'] = '/kaggle/tmp/train/study/' + temp['id'] + '.png'

# print(temp.shape)
# temp.head()

In [ ]:
class SIIMCOVID19TestData(Dataset):
    def __init__(self):
        super().__init__()
        self.submission_df = pd.read_csv("/kaggle/input/siim-covid19-detection/train_study_level.csv")
        self.submission_df['StudyInstanceUID'] = self.submission_df['id'].apply(lambda x: x.replace('_study', ''))
        self.submission_df = self.submission_df[~self.submission_df['StudyInstanceUID'].str.endswith('_image')]
        
        TEST_DIR = "/kaggle/input/siim-covid19-detection/train/"

        # Make a path folder
        paths = []
        for instance_id in tqdm(self.submission_df['StudyInstanceUID']):
            paths.append(glob.glob(os.path.join(TEST_DIR, instance_id +"/*/*"))[0])

        self.submission_df['path'] = paths
        
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.submission_df)

    def __getitem__(self,item):
        image_id = self.submission_df['StudyInstanceUID'].values[item]
        
        image_path = self.submission_df['path'].values[item]
        image = dicom2array(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = cv2.resize(image, Config.image_size)
        
        image = albumentations.Compose([albumentations.Normalize()])(image=image)['image']
        
        image = torch.tensor(image)
        
        return {
                "x":image
                }

    

test_dataset = SIIMCOVID19TestData()

test_loader = DataLoader(test_dataset,
                        batch_size=32,
                        num_workers=8)

# Load models - Colab

In [ ]:
!mkdir -p /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/resnet50-19c8e357.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/resnet18-5c106cde.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b0_ra-3dd342df.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b2_ra-bcdf34b7.pth /root/.cache/torch/hub/checkpoints/
!cp ../input/litmodels-bolts/efficientnet_b3_ra2-cf984f9c.pth /root/.cache/torch/hub/checkpoints/

In [ ]:
criterion = nn.CrossEntropyLoss()
softmax = nn.Softmax()

In [ ]:
ckpt_path1 = '../input/siimcovid19trainedmodelsstudyclassification/resnet50-siim-covid19-13-0.4999.ckpt'

resnet50_model = lit.LitResNet50.load_from_checkpoint(checkpoint_path = ckpt_path1)
resnet50_model = resnet50_model.to("cuda")
resnet50_model.eval()
resnet50_model.freeze()

In [ ]:
ckpt_path2 = '../input/siimcovid19trainedmodelsstudyclassification/resnet18-siim-covid19-33-0.4981.ckpt'

resnet18_model = lit.LitResNet18.load_from_checkpoint(checkpoint_path = ckpt_path2)
resnet18_model = resnet18_model.to("cuda")
resnet18_model.eval()
resnet18_model.freeze()

In [ ]:
ckpt_path3 = '../input/siimcovid19trainedmodelsstudyclassification/resnet18-siim-covid19-27-0.5138.ckpt'

resnet18_model2 = lit.LitResNet18_2.load_from_checkpoint(checkpoint_path = ckpt_path3)
resnet18_model2 = resnet18_model2.to("cuda")
resnet18_model2.eval()
resnet18_model2.freeze()

In [ ]:
ckpt_path4 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb0-siim-covid19-4-0.5119.ckpt'

effnetb0_model = lit.LitEffNetB0.load_from_checkpoint(checkpoint_path = ckpt_path4)
effnetb0_model = effnetb0_model.to("cuda")
effnetb0_model.eval()
effnetb0_model.freeze()

In [ ]:
ckpt_path5 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb2a-siim-covid19-14-0.4894.ckpt'

effnetb2a_model = lit.LitEffNetB2a.load_from_checkpoint(checkpoint_path = ckpt_path5)
effnetb2a_model = effnetb2a_model.to("cuda")
effnetb2a_model.eval()
effnetb2a_model.freeze()

In [ ]:
ckpt_path6 = '../input/siimcovid19trainedmodelsstudyclassification/effnetb3a-siim-covid19-26-0.4978.ckpt'

effnetb3a_model = lit.LitEffNetB3a.load_from_checkpoint(checkpoint_path = ckpt_path6)
effnetb3a_model = effnetb3a_model.to("cuda")
effnetb3a_model.eval()
effnetb3a_model.freeze()

# Inference - Colab

In [ ]:
preds_resnet50 = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet50_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet50.extend(y_hat.cpu().detach().numpy().tolist())

In [ ]:
preds_resnet18 = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet18_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet18.extend(y_hat.cpu().detach().numpy().tolist())

In [ ]:
preds_resnet18_2 = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = resnet18_model2(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_resnet18_2.extend(y_hat.cpu().detach().numpy().tolist())

In [ ]:
preds_effnetb0 = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb0_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb0.extend(y_hat.cpu().detach().numpy().tolist())

In [ ]:
preds_effnetb2a = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb2a_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb2a.extend(y_hat.cpu().detach().numpy().tolist())

In [ ]:
preds_effnetb3a = []
for ind, data in tqdm(enumerate(test_loader),total = len(test_loader)):
    y_hat = effnetb3a_model(data["x"].to("cuda"))
    y_hat = softmax(y_hat)
    preds_effnetb3a.extend(y_hat.cpu().detach().numpy().tolist())

# Inference - Efnb7

In [ ]:
import numpy as np 
import pandas as pd

df = pd.read_csv('../input/siim-covid19-detection/train_study_level.csv')
id_laststr_list  = []
for i in range(df.shape[0]):
    id_laststr_list.append(df.loc[i,'id'][-1])
df['id_last_str'] = id_laststr_list

study_len = df[df['id_last_str'] == 'y'].shape[0]

In [ ]:
import os

import efficientnet.tfkeras as efn
import numpy as np
import pandas as pd
import tensorflow as tf

def auto_select_accelerator():
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(tpu)
        tf.tpu.experimental.initialize_tpu_system(tpu)
        strategy = tf.distribute.experimental.TPUStrategy(tpu)
        print("Running on TPU:", tpu.master())
    except ValueError:
        strategy = tf.distribute.get_strategy()
    print(f"Running on {strategy.num_replicas_in_sync} replicas")

    return strategy


def build_decoder(with_labels=True, target_size=(300, 300), ext='jpg'):
    def decode(path):
        file_bytes = tf.io.read_file(path)
        if ext == 'png':
            img = tf.image.decode_png(file_bytes, channels=3)
        elif ext in ['jpg', 'jpeg']:
            img = tf.image.decode_jpeg(file_bytes, channels=3)
        else:
            raise ValueError("Image extension not supported")

        img = tf.cast(img, tf.float32) / 255.0
        img = tf.image.resize(img, target_size)

        return img

    def decode_with_labels(path, label):
        return decode(path), label

    return decode_with_labels if with_labels else decode


def build_augmenter(with_labels=True):
    def augment(img):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        return img

    def augment_with_labels(img, label):
        return augment(img), label

    return augment_with_labels if with_labels else augment


def build_dataset(paths, labels=None, bsize=32, cache=True,
                  decode_fn=None, augment_fn=None,
                  augment=True, repeat=True, shuffle=1024, 
                  cache_dir=""):
    if cache_dir != "" and cache is True:
        os.makedirs(cache_dir, exist_ok=True)

    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter(labels is not None)

    AUTO = tf.data.experimental.AUTOTUNE
    slices = paths if labels is None else (paths, labels)

    dset = tf.data.Dataset.from_tensor_slices(slices)
    dset = dset.map(decode_fn, num_parallel_calls=AUTO)
    dset = dset.cache(cache_dir) if cache else dset
    dset = dset.map(augment_fn, num_parallel_calls=AUTO) if augment else dset
    dset = dset.repeat() if repeat else dset
    dset = dset.shuffle(shuffle) if shuffle else dset
    dset = dset.batch(bsize).prefetch(AUTO)

    return dset

#COMPETITION_NAME = "siim-cov19-test-img512-study-600"

In [ ]:
strategy = auto_select_accelerator()
BATCH_SIZE = strategy.num_replicas_in_sync * 16

IMSIZE = (224, 240, 260, 300, 380, 456, 528, 600)

#load_dir = f"/kaggle/input/{COMPETITION_NAME}/"
sub_df = pd.read_csv('../input/siim-covid19-detection/train_study_level.csv')
sub_df = sub_df[:study_len]
test_paths = f'/kaggle/tmp/{split}/study/' + sub_df['id'] +'.png'

sub_df['negative'] = 0
sub_df['typical'] = 0
sub_df['indeterminate'] = 0
sub_df['atypical'] = 0

In [ ]:
label_cols = sub_df.columns[2:]

test_decoder = build_decoder(with_labels=False, target_size=(IMSIZE[7], IMSIZE[7]), ext='png')
dtest = build_dataset(
    test_paths, bsize=BATCH_SIZE, repeat=False, 
    shuffle=False, augment=False, cache=False,
    decode_fn=test_decoder
)


In [ ]:
with strategy.scope():
    
    models = []
    
    models0 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model0.h5'
    )
    models1 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model1.h5'
    )
    models2 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model2.h5'
    )
    models3 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model3.h5'
    )
    models4 = tf.keras.models.load_model(
        '../input/siim-covid19-efnb7-train-study/model4.h5'
    )
    
    models.append(models0)
    models.append(models1)
    models.append(models2)
    models.append(models3)
    models.append(models4)


In [ ]:
preds_efnb7_1 = models0.predict(dtest, verbose=1)
preds_efnb7_2 = models1.predict(dtest, verbose=1)
preds_efnb7_3 = models2.predict(dtest, verbose=1)
preds_efnb7_4 = models3.predict(dtest, verbose=1)
preds_efnb7_5 = models4.predict(dtest, verbose=1)

# Create training dataset

In [ ]:
train_image = pd.read_csv("/kaggle/input/siim-covid19-detection/train_image_level.csv")
train_study = pd.read_csv("/kaggle/input/siim-covid19-detection/train_study_level.csv")

TRAIN_DIR = "/kaggle/input/siim-covid19-detection/train/"
train_study['StudyInstanceUID'] = train_study['id'].apply(lambda x: x.replace('_study', ''))
train = train_image.merge(train_study, on='StudyInstanceUID')

# Make a path folder
paths = []
for instance_id in tqdm(train['StudyInstanceUID']):
    paths.append(glob.glob(os.path.join(TRAIN_DIR, instance_id +"/*/*"))[0])

train['path'] = paths

train['image_file'] = train['path'].apply(lambda x: x.split('/')[-1].replace('.dcm','') + '.jpg')

train = train.drop(['id_x', 'id_y'], axis=1)

In [ ]:
labels = []
for instance in tqdm(train['StudyInstanceUID'], total=len(train)):
    label = train[train['StudyInstanceUID'] == instance].values.tolist()[0][3:7].index(1)
    labels.append(label)

train['class'] = labels

train.head()

In [ ]:
train[train['StudyInstanceUID']=='0fd2db233deb'].drop_duplicates()

In [ ]:
def predCols(model, preds):
    preds0, preds1, preds2, preds3 = [], [], [], []
    
    for i in range(len(preds)):
        preds0.append(preds[i][0])
        preds1.append(preds[i][1])
        preds2.append(preds[i][2])
        preds3.append(preds[i][3])

    col0 = f"preds_{model}_0"
    col1 = f"preds_{model}_1"
    col2 = f"preds_{model}_2"
    col3 = f"preds_{model}_3"
    
    preds_train_df[col0] = preds0
    preds_train_df[col1] = preds1
    preds_train_df[col2] = preds2
    preds_train_df[col3] = preds3

In [ ]:
preds_train_df = train[['StudyInstanceUID','path','image_file','class']].drop_duplicates().copy()

predCols('resnet50',preds_resnet50)
predCols('resnet18',preds_resnet18)
predCols('resnet18_2',preds_resnet18_2)
predCols('effnetb0',preds_effnetb0)
predCols('effnetb2a',preds_effnetb2a)
predCols('effnetb3a',preds_effnetb3a)
predCols('efnb7_1',preds_efnb7_1)
predCols('efnb7_2',preds_efnb7_2)
predCols('efnb7_3',preds_efnb7_3)
predCols('efnb7_4',preds_efnb7_4)
predCols('efnb7_5',preds_efnb7_5)

print(preds_train_df.shape)
preds_train_df.head()

In [ ]:
preds_train_df.tail()

In [ ]:
preds_train_df.to_csv('/kaggle/working/DL_preds.csv',index=False)